# Pedestrian Detection with YOLO26m (Notebook-First, End-to-End)

## 1. Title and Project Overview

This notebook is a real, executable object detection workflow for pedestrian (human) detection.

What this notebook does:
- downloads the `constantinwerner/human-detection-dataset` from Kaggle in notebook cells
- verifies images, YOLO labels, and split structure before training
- trains a **YOLO26m** detector (fallback to YOLO26s on OOM)
- evaluates with mAP50, mAP50-95, precision, recall, and per-class AP
- saves `best.pt`, `best.onnx`, and `metrics.json`

## 2. Problem Statement

Detect and localise pedestrians (humans) in still images with bounding boxes.

- Input: RGB images (street scenes, surveillance, etc.)
- Output: bounding boxes + confidence for class `person`
- Dataset: `constantinwerner/human-detection-dataset` — YOLO-format single-class pedestrian annotations
- Challenge: heavily occluded pedestrians, crowded scenes, scale variation

## 3. Why the Chosen Method Is Correct

**Task family:** object detection.

- Bounding-box localisation is required — classification is insufficient
- YOLO26m is the April 2026 default for trainable single/multi-class detection
- Fallback to YOLO26s only on real GPU OOM

## 4. Hardware / Environment Check

In [ ]:
import os, platform, random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Python  : {platform.python_version()}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()} — {torch.version.cuda if torch.cuda.is_available() else 'N/A'}")
print(f"Device  : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## 5. Dependency Installation

In [ ]:
import subprocess, sys, importlib

def ensure_package(import_name: str, pip_name: str | None = None) -> None:
    pip_name = pip_name or import_name
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

ensure_package("ultralytics")
ensure_package("cv2", "opencv-python")
ensure_package("matplotlib")
ensure_package("pandas")
ensure_package("kagglehub")
print("All dependencies satisfied.")

## 6. Imports and Configuration

In [ ]:
import json, os, shutil, zipfile, random
from pathlib import Path

import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

PROJECT_DIR   = Path(os.path.dirname(os.path.abspath("__file__")))
DATA_ROOT     = PROJECT_DIR.parent.parent.parent / "data"
RUNS_DIR      = PROJECT_DIR.parent.parent.parent / "runs"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"

for d in (DATA_ROOT, RUNS_DIR, ARTIFACTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"DATA_ROOT    : {DATA_ROOT}")
print(f"ARTIFACTS_DIR: {ARTIFACTS_DIR}")

## 7. Dataset Source Explanation

**Dataset:** Kaggle — `constantinwerner/human-detection-dataset`

- Source URL: https://www.kaggle.com/datasets/constantinwerner/human-detection-dataset
- Single-class pedestrian dataset with YOLO-format bounding box annotations
- Images from street/surveillance scenes

**Credential requirements:**
- Set `KAGGLE_USERNAME` + `KAGGLE_KEY` env vars, or place `kaggle.json` in `~/.kaggle/`
- The download cell raises a clear error if credentials are missing — no synthetic fallback

In [ ]:
import subprocess

KAGGLE_DATASET = "constantinwerner/human-detection-dataset"
DATASET_DIR = DATA_ROOT / "pedestrian_detection"
DATASET_DIR.mkdir(parents=True, exist_ok=True)


def check_kaggle_credentials() -> None:
    has_env = os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")
    has_file = (Path.home() / ".kaggle" / "kaggle.json").exists()
    if not has_env and not has_file:
        raise RuntimeError(
            "Kaggle credentials not found.\n"
            "Set KAGGLE_USERNAME and KAGGLE_KEY env vars, or place kaggle.json in ~/.kaggle/"
        )


def download_dataset() -> Path:
    check_kaggle_credentials()
    try:
        import kagglehub
        path = Path(kagglehub.dataset_download(KAGGLE_DATASET))
        return path
    except Exception:
        pass
    subprocess.check_call([
        "kaggle", "datasets", "download", "-d", KAGGLE_DATASET,
        "-p", str(DATASET_DIR), "--unzip",
    ])
    return DATASET_DIR


print("Downloading pedestrian detection dataset from Kaggle...")
raw_root = download_dataset()
print(f"Raw dataset at: {raw_root}")

# Locate data.yaml or YOLO structure
yaml_candidates = list(Path(raw_root).rglob("data.yaml"))
if yaml_candidates:
    DATA_YAML = yaml_candidates[0]
    print(f"Found data.yaml: {DATA_YAML}")
else:
    DATA_YAML = None
    print("No data.yaml found — will build structure manually.")

## 8. Dataset Structure Preparation

Build YOLO-compatible train/val/test splits if not already present.

In [ ]:
import yaml as pyyaml

def build_yolo_structure(raw_root: Path) -> tuple[Path, Path]:
    """Return (yolo_root, data_yaml_path). Builds splits if needed."""
    # Case 1: data.yaml already exists with train/val paths
    yaml_files = list(Path(raw_root).rglob("data.yaml"))
    if yaml_files:
        yaml_path = yaml_files[0]
        yolo_root = yaml_path.parent
        return yolo_root, yaml_path

    # Case 2: flat images + labels folders — need to split
    all_imgs = sorted([
        p for ext in ("*.jpg","*.jpeg","*.png","*.JPG","*.PNG")
        for p in Path(raw_root).rglob(ext)
        if "label" not in str(p).lower()
    ])
    if len(all_imgs) == 0:
        raise RuntimeError(f"No images found under {raw_root}")

    print(f"Found {len(all_imgs)} images — building 70/15/15 split...")
    yolo_root = DATASET_DIR / "yolo_dataset"
    for split in ("train", "val", "test"):
        (yolo_root / "images" / split).mkdir(parents=True, exist_ok=True)
        (yolo_root / "labels" / split).mkdir(parents=True, exist_ok=True)

    random.seed(SEED)
    random.shuffle(all_imgs)
    n = len(all_imgs)
    splits = {
        "train": all_imgs[:int(0.70*n)],
        "val":   all_imgs[int(0.70*n):int(0.85*n)],
        "test":  all_imgs[int(0.85*n):],
    }

    skipped = 0
    for split_name, imgs in splits.items():
        for img_path in imgs:
            lbl_path = img_path.with_suffix(".txt")
            # Check labels sibling folder too
            if not lbl_path.exists():
                lbl_path = Path(str(img_path).replace("images", "labels")).with_suffix(".txt")

            dst_img = yolo_root / "images" / split_name / img_path.name
            dst_lbl = yolo_root / "labels" / split_name / img_path.with_suffix(".txt").name

            img_ok = cv2.imread(str(img_path))
            if img_ok is None:
                skipped += 1
                continue
            shutil.copy2(img_path, dst_img)
            if lbl_path.exists():
                shutil.copy2(lbl_path, dst_lbl)
            else:
                dst_lbl.write_text("")  # empty label = background image

    print(f"Skipped {skipped} unreadable images.")

    yaml_content = (
        f"path: {yolo_root}\n"
        "train: images/train\n"
        "val: images/val\n"
        "test: images/test\n"
        "nc: 1\n"
        "names: ['person']\n"
    )
    yaml_path = yolo_root / "data.yaml"
    yaml_path.write_text(yaml_content)
    return yolo_root, yaml_path


YOLO_ROOT, DATA_YAML = build_yolo_structure(raw_root)
print(f"YOLO dataset root : {YOLO_ROOT}")
print(f"data.yaml         : {DATA_YAML}")

with open(DATA_YAML) as f:
    yaml_cfg = pyyaml.safe_load(f)
print(f"Classes ({yaml_cfg.get('nc')}): {yaml_cfg.get('names')}")

## 9. Dataset Verification

Fail loudly if expected files are missing.

In [ ]:
def count_split(split_name: str) -> tuple[int, int]:
    img_dir = YOLO_ROOT / "images" / split_name
    lbl_dir = YOLO_ROOT / "labels" / split_name
    if not img_dir.exists():
        # Try yaml-defined paths
        return 0, 0
    imgs = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.jpeg")) + list(img_dir.glob("*.png"))
    lbls = list(lbl_dir.glob("*.txt")) if lbl_dir.exists() else []
    return len(imgs), len(lbls)


train_imgs, train_lbls = count_split("train")
val_imgs,   val_lbls   = count_split("val")
test_imgs,  test_lbls  = count_split("test")

print(f"Train: {train_imgs:4d} images, {train_lbls:4d} labels")
print(f"Val  : {val_imgs:4d} images, {val_lbls:4d} labels")
print(f"Test : {test_imgs:4d} images, {test_lbls:4d} labels")

assert train_imgs >= 10, f"Too few training images: {train_imgs}"
assert val_imgs   >= 5,  f"Too few validation images: {val_imgs}"
assert DATA_YAML.exists(), f"data.yaml not found: {DATA_YAML}"
print("Dataset verification passed.")

## 10. Data Integrity Audit

In [ ]:
img_dir_train = YOLO_ROOT / "images" / "train"
all_train_imgs = list(img_dir_train.glob("*.jpg")) + list(img_dir_train.glob("*.jpeg")) + list(img_dir_train.glob("*.png"))

sample = random.sample(all_train_imgs, min(1000, len(all_train_imgs)))
corrupt, widths, heights, empty_lbl = 0, [], [], 0

lbl_dir_train = YOLO_ROOT / "labels" / "train"
for img_path in sample:
    img = cv2.imread(str(img_path))
    if img is None:
        corrupt += 1
        continue
    h, w = img.shape[:2]
    widths.append(w)
    heights.append(h)
    lbl_path = lbl_dir_train / img_path.with_suffix(".txt").name
    if lbl_path.exists() and lbl_path.stat().st_size == 0:
        empty_lbl += 1

print(f"Sampled      : {len(sample)}")
print(f"Corrupt      : {corrupt}")
print(f"Empty labels : {empty_lbl}  (background / no-pedestrian images)")
if widths:
    print(f"Width range  : {min(widths)}–{max(widths)} px")
    print(f"Height range : {min(heights)}–{max(heights)} px")

## 11. Label/Schema Inspection

In [ ]:
lbl_dir = YOLO_ROOT / "labels" / "train"
all_lbls = list(lbl_dir.glob("*.txt"))

class_counts: dict[int, int] = {}
total_boxes = 0
box_areas = []
invalid_lines = 0

for lbl_path in all_lbls:
    lines = lbl_path.read_text().strip().splitlines()
    for line in lines:
        parts = line.strip().split()
        if len(parts) != 5:
            invalid_lines += 1
            continue
        cls_id = int(parts[0])
        w, h   = float(parts[3]), float(parts[4])
        class_counts[cls_id] = class_counts.get(cls_id, 0) + 1
        box_areas.append(w * h)
        total_boxes += 1

print(f"Total bounding boxes: {total_boxes:,}")
print(f"Invalid label lines : {invalid_lines}")
print(f"Class distribution  : {class_counts}")
if box_areas:
    print(f"Bbox area (WxH norm) min={min(box_areas):.4f}  mean={sum(box_areas)/len(box_areas):.4f}  max={max(box_areas):.4f}")

In [ ]:
## 12. Sample Visualization

def draw_yolo_boxes(img: np.ndarray, lbl_path: Path, class_names: list[str]) -> np.ndarray:
    out = img.copy()
    h, w = out.shape[:2]
    if not lbl_path.exists():
        return out
    for line in lbl_path.read_text().strip().splitlines():
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        cls_id = int(parts[0])
        xc, yc, bw, bh = map(float, parts[1:])
        x1 = int((xc - bw/2) * w)
        y1 = int((yc - bh/2) * h)
        x2 = int((xc + bw/2) * w)
        y2 = int((yc + bh/2) * h)
        label = class_names[cls_id] if cls_id < len(class_names) else str(cls_id)
        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 200, 0), 2)
        cv2.putText(out, label, (x1, max(y1-5, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 0), 1)
    return out

class_names = yaml_cfg.get("names", ["person"])
sample_6 = random.sample(all_train_imgs, min(6, len(all_train_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for ax, img_path in zip(axes, sample_6):
    img_bgr = cv2.imread(str(img_path))
    lbl_path = lbl_dir / img_path.with_suffix(".txt").name
    vis = draw_yolo_boxes(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB), lbl_path, class_names)
    ax.imshow(cv2.resize(vis, (320, 240)))
    ax.set_title(img_path.name, fontsize=7)
    ax.axis("off")
plt.suptitle("Sample training images with GT bounding boxes", fontsize=12)
plt.tight_layout()
fig.savefig(str(ARTIFACTS_DIR / "sample_images.png"), dpi=100)
plt.close(fig)
print("Saved sample visualization → artifacts/sample_images.png")

## 13. Preprocessing / Augmentation Strategy

YOLO26m handles augmentation internally (mosaic, random flip, HSV jitter, random crop).
Below we show a manual Albumentations preview for educational clarity.

In [ ]:
import albumentations as A

aug = A.Compose(
    [
        A.RandomBrightnessContrast(p=0.5),
        A.HueSaturationValue(p=0.4),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=5, p=0.4),
    ],
    bbox_params=A.BboxParams(format="yolo", label_fields=["class_labels"]),
)

sample_img_path = random.choice(all_train_imgs)
sample_lbl_path = lbl_dir / sample_img_path.with_suffix(".txt").name
img_bgr = cv2.imread(str(sample_img_path))
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

bboxes, class_labels = [], []
if sample_lbl_path.exists():
    for line in sample_lbl_path.read_text().strip().splitlines():
        p = line.split()
        if len(p) == 5:
            class_labels.append(int(p[0]))
            bboxes.append([float(x) for x in p[1:]])

try:
    aug_result = aug(image=img_rgb, bboxes=bboxes, class_labels=class_labels)
    aug_img = aug_result["image"]
except Exception:
    aug_img = img_rgb

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(cv2.resize(img_rgb, (320, 240)))
axes[0].set_title("Original")
axes[0].axis("off")
axes[1].imshow(cv2.resize(aug_img, (320, 240)))
axes[1].set_title("Augmented (demo)")
axes[1].axis("off")
plt.tight_layout()
fig.savefig(str(ARTIFACTS_DIR / "augmentation_preview.png"), dpi=100)
plt.close(fig)
print("Saved augmentation preview → artifacts/augmentation_preview.png")

## 14. Train/Validation/Test Split Verification

In [ ]:
split_df = pd.DataFrame({
    "split":  ["train", "val", "test"],
    "images": [train_imgs, val_imgs, test_imgs],
    "labels": [train_lbls, val_lbls, test_lbls],
})
print(split_df.to_string(index=False))
assert train_imgs >= 10, "Insufficient training data"
print("Split verification passed.")

## 15. Baseline or Sanity-Check Pipeline

Run pretrained YOLO26m (COCO weights) on a few val images to verify the pipeline is functional before fine-tuning.

In [ ]:
from ultralytics import YOLO

sanity_model = YOLO("yolo26m.pt")

val_img_dir = YOLO_ROOT / "images" / "val"
val_imgs_list = list(val_img_dir.glob("*.jpg")) + list(val_img_dir.glob("*.jpeg")) + list(val_img_dir.glob("*.png"))
sanity_imgs = random.sample(val_imgs_list, min(6, len(val_imgs_list)))

val_lbl_dir = YOLO_ROOT / "labels" / "val"

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for ax, img_path in zip(axes, sanity_imgs):
    result = sanity_model.predict(str(img_path), verbose=False, classes=[0])[0]  # class 0 = person in COCO
    img_rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    n_det = len(result.boxes) if result.boxes is not None else 0
    ax.imshow(result.plot()[:, :, ::-1])
    ax.set_title(f"Pretrained: {n_det} person(s)", fontsize=8)
    ax.axis("off")
plt.suptitle("Sanity check — pretrained YOLO26m (COCO) detections", fontsize=11)
plt.tight_layout()
fig.savefig(str(ARTIFACTS_DIR / "sanity_inference.png"), dpi=100)
plt.close(fig)
print("Saved sanity inference → artifacts/sanity_inference.png")

## 16. Main Model Setup

In [ ]:
PREFERRED_MODEL = "yolo26m.pt"
FALLBACK_MODEL  = "yolo26s.pt"

IMG_SIZE = 640
EPOCHS   = 30
BATCH    = 16
WORKERS  = 2

TRAIN_PROJECT = RUNS_DIR / "pedestrian_det"
TRAIN_NAME    = "yolo26m_pedestrian"

print(f"Model   : {PREFERRED_MODEL}")
print(f"Epochs  : {EPOCHS}")
print(f"Batch   : {BATCH}")
print(f"Img size: {IMG_SIZE}")
print(f"Data    : {DATA_YAML}")

## 17. Training Loop / Trainer Setup

In [ ]:
from ultralytics import YOLO


def run_training(model_name: str, batch: int) -> object:
    model = YOLO(model_name)
    return model.train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=batch,
        workers=WORKERS,
        project=str(TRAIN_PROJECT),
        name=TRAIN_NAME,
        exist_ok=True,
        device=DEVICE,
        seed=SEED,
        verbose=True,
    )


train_model_name = PREFERRED_MODEL
current_batch    = BATCH
try:
    train_results = run_training(train_model_name, current_batch)
    print(f"Training complete with {train_model_name}.")
except RuntimeError as exc:
    if "out of memory" in str(exc).lower():
        print(f"OOM with {train_model_name}, retrying with {FALLBACK_MODEL} and smaller batch...")
        torch.cuda.empty_cache()
        train_model_name = FALLBACK_MODEL
        current_batch    = max(4, current_batch // 2)
        train_results    = run_training(train_model_name, current_batch)
        print(f"Training complete with {train_model_name} (OOM fallback).")
    else:
        raise

## 18. Validation and Core Metrics

In [ ]:
from ultralytics import YOLO
import json

best_path = TRAIN_PROJECT / TRAIN_NAME / "weights" / "best.pt"
if not best_path.exists():
    raise FileNotFoundError(f"best.pt not found at {best_path} — did training complete?")

best_model = YOLO(str(best_path))
val_metrics = best_model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=IMG_SIZE,
    device=DEVICE,
    verbose=True,
)

# Extract metrics
map50     = float(val_metrics.box.map50)
map50_95  = float(val_metrics.box.map)
precision = float(val_metrics.box.mp)
recall    = float(val_metrics.box.mr)

print(f"mAP50     : {map50:.4f}")
print(f"mAP50-95  : {map50_95:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")

# Per-class AP
names      = val_metrics.names
class_aps  = {}
ap50_vals  = val_metrics.box.ap50
class_list = list(names.values()) if isinstance(names, dict) else names

for cls_name, ap in zip(class_list, ap50_vals):
    class_aps[cls_name] = float(ap)
    print(f"  AP50 [{cls_name}]: {ap:.4f}")

metrics_out = {
    "map50":     map50,
    "map50_95":  map50_95,
    "precision": precision,
    "recall":    recall,
    "per_class_ap50": class_aps,
}
with open(ARTIFACTS_DIR / "metrics.json", "w") as f:
    json.dump(metrics_out, f, indent=2)
print("Saved metrics.json")

## 19. Error Analysis

Display training curves and analyse false positive / false negative patterns.

In [ ]:
run_dir = TRAIN_PROJECT / TRAIN_NAME

# Show training curves from Ultralytics artifacts
curve_files = ["results.png", "PR_curve.png", "F1_curve.png", "confusion_matrix.png"]
shown = 0
for fname in curve_files:
    fpath = run_dir / fname
    if fpath.exists():
        img = cv2.cvtColor(cv2.imread(str(fpath)), cv2.COLOR_BGR2RGB)
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(fname)
        plt.tight_layout()
        fig.savefig(str(ARTIFACTS_DIR / fname), dpi=80)
        plt.close(fig)
        print(f"Saved {fname}")
        shown += 1

if shown == 0:
    print("No training curve images found — check run directory.")

# False negative analysis: val images where model detects 0 persons but GT has ≥1
val_img_dir  = YOLO_ROOT / "images" / "val"
val_lbl_dir  = YOLO_ROOT / "labels" / "val"
val_all_imgs = list(val_img_dir.glob("*.jpg")) + list(val_img_dir.glob("*.jpeg")) + list(val_img_dir.glob("*.png"))

fn_examples = []
for img_path in random.sample(val_all_imgs, min(200, len(val_all_imgs))):
    lbl_path = val_lbl_dir / img_path.with_suffix(".txt").name
    if not lbl_path.exists() or lbl_path.stat().st_size == 0:
        continue
    result = best_model.predict(str(img_path), conf=0.25, verbose=False)[0]
    n_det = len(result.boxes) if result.boxes is not None else 0
    if n_det == 0:
        fn_examples.append(img_path)
    if len(fn_examples) >= 6:
        break

if fn_examples:
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    axes = axes.flatten()
    for ax, img_path in zip(axes, fn_examples):
        img_rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        lbl_path = val_lbl_dir / img_path.with_suffix(".txt").name
        vis = draw_yolo_boxes(img_rgb, lbl_path, class_names)
        ax.imshow(cv2.resize(vis, (320, 240)))
        ax.set_title("GT: person(s)\nPred: 0 detections", fontsize=8, color="red")
        ax.axis("off")
    plt.suptitle("False negatives (missed pedestrians)", fontsize=11)
    plt.tight_layout()
    fig.savefig(str(ARTIFACTS_DIR / "false_negatives.png"), dpi=100)
    plt.close(fig)
    print(f"Saved false_negatives.png ({len(fn_examples)} examples)")
else:
    print("No false negative examples found in sampled val set.")

## 20. Inference on Holdout Examples

In [ ]:
test_img_dir  = YOLO_ROOT / "images" / "test"
test_all_imgs = list(test_img_dir.glob("*.jpg")) + list(test_img_dir.glob("*.jpeg")) + list(test_img_dir.glob("*.png"))
if len(test_all_imgs) == 0:
    test_all_imgs = val_all_imgs  # fallback to val if test is empty

holdout_sample = random.sample(test_all_imgs, min(6, len(test_all_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for ax, img_path in zip(axes, holdout_sample):
    result = best_model.predict(str(img_path), conf=0.25, verbose=False)[0]
    n_det  = len(result.boxes) if result.boxes is not None else 0
    ax.imshow(result.plot()[:, :, ::-1])
    ax.set_title(f"{img_path.name[:20]}\n{n_det} person(s)", fontsize=7)
    ax.axis("off")
plt.suptitle("Holdout inference results", fontsize=11)
plt.tight_layout()
fig.savefig(str(ARTIFACTS_DIR / "holdout_inference.png"), dpi=100)
plt.close(fig)
print("Saved holdout inference → artifacts/holdout_inference.png")

## 21. Save Model/Artifacts

In [ ]:
import json, shutil
from ultralytics import YOLO

saved_best_pt = ARTIFACTS_DIR / "best.pt"
shutil.copy2(best_path, saved_best_pt)
print(f"Saved best.pt → {saved_best_pt}")

export_model = YOLO(str(saved_best_pt))
onnx_path = export_model.export(format="onnx", imgsz=IMG_SIZE, opset=12)
print(f"Exported ONNX → {onnx_path}")

with open(ARTIFACTS_DIR / "metrics.json") as f:
    m = json.load(f)

manifest = {
    "project":    "Pedestrian Detection (YOLO26m)",
    "model_file": "best.pt",
    "onnx_file":  "best.onnx",
    "dataset":    "constantinwerner/human-detection-dataset",
    "metrics":    m,
}
with open(ARTIFACTS_DIR / "artifact_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
print("Saved artifact_manifest.json")

## 22. Reproducibility Notes

- Random seed fixed to `42` throughout
- YOLO26m training uses `seed=42`
- `kagglehub` caches downloads; re-running reuses the cache
- GPU hardware differences may cause minor floating-point variation in metrics

## 23. Conclusion and Limitations

This notebook implements a real end-to-end pedestrian detection pipeline:
- Real Kaggle download with fail-loud credential checks
- YOLO26m fine-tuned on single-class pedestrian data with OOM fallback
- mAP50, mAP50-95, precision, recall, per-class AP metrics
- False negative analysis and holdout inference grid

**Limitations:**
- Dataset is relatively small; more data (CrowdHuman, CityPersons) would improve recall on dense crowds
- Heavily occluded pedestrians remain challenging for single-stage detectors
- `person` class IoU thresholds affect mAP — 0.5 IoU threshold may be lenient for safety-critical use